# Module Validate — Vérification du contrat de données

Ce notebook analyse et documente le script `validate_clean.py`.

**Rôle :** Vérifier que le fichier CSV unique généré par la transformation respecte strictement le contrat de données avant son intégration dans le Data Warehouse.
**Principes :**
*   **Complétude :** Toutes les colonnes requises doivent être présentes, et aucun champ clé ne doit être vide.
*   **Unicité :** Une seule ligne autorisée par couple (ville, timestamp).
*   **Tri chronologique :** Les données d'une même ville doivent avancer dans le temps.
*   **Cohérence métier :** Les valeurs d'AQI doivent être comprises dans l'échelle officielle (1-5).

In [5]:
# 0. Configuration Colab
import sys
import pandas as pd

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    CLEAN_FILE = "qualite_air.csv"

    CITIES = [
        {"name": "Antananarivo"},
        {"name": "Beijing"},
        {"name": "Nairobi"},
        {"name": "Paris"},
        {"name": "Mumbai"}
    ]
else:
    from config import CLEAN_FILE, CITIES

REQUIRED_COLS = [
    "ville", "pays", "latitude", "longitude", "timestamp_utc", "aqi",
    "co", "no", "no2", "o3", "so2", "pm2_5", "pm10", "nh3",
]

EXPECTED_CITIES = {c["name"] for c in CITIES}
print("Configuration terminée ✓")

Configuration terminée ✓


### 1. Analyse de la fonction principale

`validate() -> list[str]`
Ouvre le fichier clean généré et exécute une batterie de 6 tests de qualité.
Retourne une liste contenant toutes les erreurs rencontrées, sans s'arrêter au premier échec.

In [6]:
def validate() -> list[str]:
    errors = []

    try:
        df = pd.read_csv(CLEAN_FILE)
    except FileNotFoundError:
        return [f"Erreur : Le fichier {CLEAN_FILE} est introuvable."]

    missing_cols = set(REQUIRED_COLS) - set(df.columns)
    if missing_cols:
        errors.append(f"Colonnes manquantes : {missing_cols}")
        return errors

    key_cols = ["ville", "pays", "latitude", "longitude", "timestamp_utc"]
    n_missing = df[key_cols].isna().any(axis=1).sum()
    if n_missing:
        errors.append(f"{n_missing} lignes avec des champs clés manquants")

    n_dupes = df.duplicated(subset=["ville", "timestamp_utc"]).sum()
    if n_dupes:
        errors.append(f"{n_dupes} doublons sur (ville, timestamp_utc)")

    for ville, g in df.groupby("ville"):
        ts = pd.to_datetime(g["timestamp_utc"])
        if not ts.is_monotonic_increasing:
            errors.append(f"{ville} : timestamps non triés chronologiquement")

    bad_aqi = df["aqi"].dropna()
    if not bad_aqi.between(1, 5).all():
        errors.append("Valeurs d'aqi hors de l'échelle OpenWeather 1-5")

    missing_cities = EXPECTED_CITIES - set(df["ville"].unique())
    if missing_cities:
        errors.append(f"Villes absentes de clean/ : {missing_cities}")

    return errors

print("Fonction validate définie ✓")

Fonction validate définie ✓


### 2. Exécution du contrôle qualité

In [7]:
problems = validate()

if problems:
    print("❌ VALIDATION ÉCHOUÉE :")
    for p in problems:
        print(f"  - {p}")
else:
    print("✅ SUCCÈS : clean/qualite_air.csv est strictement conforme au contrat de données.")

✅ SUCCÈS : clean/qualite_air.csv est strictement conforme au contrat de données.
